## Laboratorio 6

- Diego Valenzuela 22309 
- Daniel Dubon 22233
- Nelson García Bravatti 22434
- Joaquin Puente 22296

# Task 1

1. El entorno de simulación que usarán es LunarLanderContinuous-v2 de Gymnasium, que tiene un
espacio de acción continuo de dos dimensiones representando la fuerza de dos propulsores.
Argumenten formalmente por qué Q-Learning tabular y DQN son inapropiados para este entorno. Su
argumento debe mencionar explícitamente el espacio de acción, el operador argmax, y la
representación de la política

Tanto Q-Learning tabular como DQN son algoritmos diseñados para entornos discretos por lo que su aplicación en LunarLanderContinuous-v2 es inapropiada ya que el entorno posee un espacio de acción continuo bidimensional para controlar las fuerzas de los dos propulsores. Q-Learning tabular requiere construir una tabla o matriz donde cada par de estado y acción tiene una entrada unica, como el espacio de acción es continuo existen infinitas acciones posibles y crear esta tabla es imposible sin aplicar una discretización extrema que causaria la maldición de la dimensionalidad y perdida de información. Para el caso de DQN el obstaculo principal es el operador argmax donde DQN usa redes neuronales y puede manejar estados continuos pero su ecuación de actualización de Bellman requiere aplicar el argmax sobre todas las acciones posibles para encontrar el mayor valor esperado. Evaluar el $\arg\max_a Q(s, a)$ en un espacio de acción continuo requiere resolver un problema de optimización complejo en cada iteración del agente lo cual hace que el algoritmo sea computacionalmente inviable.Finalmente esto se reduce a la representación de la politica. Los algoritmos basados en valor como Q-Learning y DQN tienen una representación de politica implicita que depende totalmente de evaluar acciones discretas individuales para elegir la mejor. En dominios de control continuo se necesitan algoritmos como Actor-Critic o REINFORCE que mantienen una representación de politica parametrizada y explicita.


---

2. Para LunarLanderContinuous-v2, la política se parametrizará como una distribución Gaussiana
𝜋𝜃(𝑎 ∣ 𝑠) = 𝒩(𝜇𝜃, (𝑠)𝜎2𝐼) donde 𝜇𝜃(𝑠) es la salida de una red neuronal. Expliquen cómo se calcula
∇𝜃 ln 𝜋𝜃 (𝐴𝑡 ∣ 𝑆𝑡) para esta parametrización específica. Desarrollen la expresión analítica del
gradiente del logaritmo de la densidad Gaussiana respecto a 𝜃, identificando qué parte depende de
𝜃 y qué parte no



Para calcular el gradiente del logaritmo de la politica primero debemos partir de la función de densidad de probabilidad de una distribución Gaussiana como la politica es $\pi_\theta(A_t \mid S_t) = \mathcal{N}(\mu_\theta(S_t), \sigma^2 I)$ su función de densidad se escribe de la siguiente manera:

$$\pi_\theta(A_t \mid S_t) = \frac{1}{\sqrt{(2\pi)^k \vert{}\sigma^2 I\vert{}}} \exp\left(-\frac{1}{2\sigma^2} \Vert{}A_t - \mu_\theta(S_t)\Vert{}^2\right)$$

Ahora aplicamos el logaritmo natural a esta formula para separar todo y el logaritmo convierte la multiplicación en suma y elimina el exponencial de la ecuación:

$$\ln \pi_\theta(A_t \mid S_t) = -\frac{k}{2} \ln(2\pi \sigma^2) - \frac{1}{2\sigma^2} \Vert{}A_t - \mu_\theta(S_t)\Vert{}^2$$

El siguiente paso es derivar esta formula con respecto a los parametros $\theta$. Al aplicar el gradiente $\nabla_\theta$ vemos que el primer termino de la normalización es una constante que no depende de $\theta$ por lo que su derivada es cero. Solo derivamos el termino de la derecha aplicando la regla de la cadena para obtener la expresión analitica final:

$$\nabla_\theta \ln \pi_\theta(A_t \mid S_t) = \frac{(A_t - \mu_\theta(S_t))}{\sigma^2} \nabla_\theta \mu_\theta(S_t)$$

Al observar este resultado podemos identificar claramente las dependencias:

Las partes que dependen de $\theta$ son la media $\mu_\theta(S_t)$ ya que es la salida directa de la red neuronal y el jacobiano $\nabla_\theta \mu_\theta(S_t)$ que representa los gradientes de la red neuronal respecto a sus propios pesos.

Las partes que no dependen de $\theta$ son la varianza $\sigma^2$ (que en esta parametrización especifica es un escalar constante que no se aprende) la acción muestreada $A_t$ y el estado del entorno $S_t$. La constante inicial de la distribución tampoco depende de $\theta$ y por eso desaparece durante la derivación.

---

3. Comparen formalmente REINFORCE con línea base y Actor-Critic en términos de sesgo y varianza del
estimador del gradiente. Para cada algoritmo identifiquen: qué usa como estimador de la ventaja 𝐴̂ 𝑡
, qué componente introduce sesgo, y qué componente introduce varianza. Predigan cuál algoritmo
esperan que converja más rápido en LunarLanderContinuous-v2 y justifiquen esa predicción.

REINFORCE con linea base utiliza el retorno real del episodio menos la predicción de la linea base como estimador de la ventaja $\hat{A}_t = G_t - V(S_t)$. En este algoritmo el estimador no tiene sesgo porque el retorno $G_t$ es una muestra real y exacta obtenida del entorno, sin embargo el componente que introduce alta varianza es justamente el uso de $G_t$ ya que calcular el retorno requiere sumar las recompensas de toda una trayectoria completa acumulando la aleatoriedad e incertidumbre de cada paso hasta el final pero por otro lado Actor-Critic utiliza el error de diferencia temporal como estimador de la ventaja $\hat{A}_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$. El componente que introduce sesgo es el uso de $V(S_{t+1})$ lo cual se conoce como bootstrapping. Al usar la predicción de la propia red neuronal del critico para estimar el futuro en lugar de usar datos reales se introduce un sesgo fuerte especialmente al inicio de la simulación cuando la red aun no sabe nada y a cambio el componente que elimina la varianza es el hecho de usar un solo paso real $R_{t+1}$ evitando toda la incertidumbre de una trayectoria completa.Para el entorno LunarLanderContinuous-v2 se prevee que Actor-Critic convergera mas rapido ya que en un espacio de acción continuo las trayectorias tienen una varianza inmensa porque existen infinitas formas de activar los propulsores y REINFORCE sufre demasiado con esta varianza y sus actualizaciones de gradiente serian muy caoticas e inestables. Actor-Critic sacrifica precisión inicial aceptando el sesgo pero al actualizar la politica en cada paso con varianza baja logra estabilizar el entrenamiento mucho mas pronto y aprender a aterrizar en una fracción del tiempo.

---

4. El entorno de exoesqueleto real tiene una restricción que LunarLanderContinuous-v2 no tiene:
las acciones deben ser suaves en el tiempo para no causar movimientos bruscos que dañen al
paciente. Argumenten cómo modificarían la función de recompensa y la parametrización de la
política para incorporar esa restricción. ¿Cambiaría eso la elección entre REINFORCE y Actor-Critic?

Para modificar la función de recompensa se agregaria un termino de penalización que castigue los cambios bruscos en las fuerzas del motor restando a la recompensa original un valor proporcional a la diferencia matematica entre la acción actual y la acción anterior. De esta forma el agente recibe menos puntos si el exoesqueleto da tirones o cambia de fuerza de un instante a otro por lo que el algoritmo se ve forzado a priorizar un movimiento fluido.

En cuanto a la parametrización de la politica la red neuronal ya no deberia calcular la fuerza absoluta del motor de forma aislada. La forma mas inteligente de incorporar esta restricción es hacer que la red neuronal calcule solamente el cambio de fuerza o delta y que ese pequeño ajuste se sume a la acción anterior u otra opcion es incluir la acción pasada directamente como parte del estado de entrada para que el modelo tome la siguiente decisión teniendo ese contexto en cuenta.

Esta modificación no cambiaria la elección entre REINFORCE y Actor-Critic sino que haria a Actor-Critic mucho mas importante porque al agregar la acción anterior al sistema y requerir suavidad en el tiempo el problema de control se vuelve mas estricto y evaluar episodios completos generaria una varianza todavia mas alta lo que destruiria el gradiente en REINFORCE. Actor-Critic sigue siendo la mejor opción porque su capacidad de estimar la ventaja paso a paso mantiene el aprendizaje estable y permite que la red se adapte a estas restricciones sin sufrir por el ruido de toda la trayectoria.